<a href="https://colab.research.google.com/github/Lingeshkumar24-code/LingeshKumar-gen-ai-foundations/blob/main/Fine_Tuning_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We are NOT training BERT from scratch. We take a pre-trained DistilBERT model and continue training it on labelled movie reviews so that it learns the Positive/Negative classification task.

What we are going to do :

Pre-trained DistilBERT
        -
Already understands language
        -
Give it labelled movie reviews
        -
"Excellent movie!" → Positive
"Very boring movie" → Negative
        -
Model adjusts its weights
        -
Fine-tuned Sentiment Model

Steps that will be followed:

IMDb Dataset
     -
Train / Test Split
     -
Tokenization
     -
Pre-trained DistilBERT
     -
Fine-Tuning
     -
Evaluation
     -
New Review → Positive / Negative

In [3]:
!pip install -q transformers datasets evaluate accelerate

We need:

datasets → get IMDb dataset

transformers → BERT/DistilBERT and tokenizer

evaluate → calculate accuracy

accelerate → helps Hugging Face run training efficiently

In [4]:
#Import Libraries
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,#Loads a pretrained Transformer model designed for classification tasks.
    TrainingArguments,#Defines how the model should be trained.
    Trainer#Handles the actual training and evaluation process.
)
import evaluate
import numpy as np

In [5]:
!pip install -U datasets huggingface_hub

In [6]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

In [7]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [8]:
print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

0--- negative
1----positive

In [9]:
#Taking a small dataset for demo
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

In [10]:
print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 500
})


In [11]:
#Loading Pre0trained DistilBert
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


This is a pre-trained Transformer model.

It has already learned general language patterns.

We are adding:
Sentiment Classification

        ↓

Negative / Positive
The num_labels=2 tells the model:

Oour classification problem has two classes.

DistilBERT is a smaller version of BERT designed to retain much of BERT's language capability with less computational cost.

In [12]:
#Model overview before fine tuning
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

In [13]:
classifier("I absolutely loved this movie!")

[{'label': 'LABEL_0', 'score': 0.5016089677810669}]

In [14]:
classifier("This movie was extremely boring.")

[{'label': 'LABEL_0', 'score': 0.5049951076507568}]

Because this model has just been loaded with a classification head that hasn't been trained for our IMDb labels, don't expect meaningful sentiment predictions yet.

In [15]:
#Tokenization
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [16]:
#Applying
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [17]:
#Inspecting
print(tokenized_train[0])

{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1, 'input_ids': [101, 2045, 2003, 2053, 7189, 2012, 2035, 2090, 3481, 3771, 1998, 6337, 2099, 2021, 1996, 2755, 2008, 2119, 2024, 2610, 2186, 2055, 6355, 6997, 1012, 6337, 2099, 3504, 15594, 2100, 1010, 3481, 3771, 350

These numbers are called:

Token IDs / Input IDs

The Transformer does not directly receive English sentences.

It receives numerical representations.

Hugging Face notes that tokenization creates model inputs such as input_ids and attention_mask
https://huggingface.co/docs/transformers/training

In [18]:
#We do not need the original text columns
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

In [19]:
print(tokenized_train.column_names)

['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [20]:
#checking accuracy
accuracy = evaluate.load("accuracy")

In [21]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

The model produces scores for:

Negative → 2.1
Positive → 4.8

We choose the class with the highest score.

So:

np.argmax(...)

means:

"Give the class with the highest score."


In [22]:
#This is where fine tuning starts
training_args = TrainingArguments(
    output_dir="./sentiment_model",
    eval_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

In [23]:
#Creating Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [24]:
#starts fine tuning process
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.496158,0.419324,0.828000
2,0.294287,0.411372,0.820000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.42305534744262696, metrics={'train_runtime': 51.9649, 'train_samples_per_second': 76.975, 'train_steps_per_second': 4.811, 'total_flos': 132467398656000.0, 'train_loss': 0.42305534744262696, 'epoch': 2.0})



Conceptually:

Movie Review
     ↓
Tokenizer
     ↓
Input IDs
     ↓
DistilBERT
     ↓
Prediction
     ↓
Compare with actual label
     ↓
Calculate Loss
     ↓
Backpropagation
     ↓
Update Weights
     ↓
Next Batch

For example:

Training example
Review:
"This movie was fantastic!"

Actual label:
1 → Positive

Model initially predicts:

Negative = 0.60
Positive = 0.40

That's wrong.

So the model calculates a loss.

Then:

Loss
 ↓
Backpropagation
 ↓
Gradients
 ↓
Weights updated

After seeing many examples, the model gradually becomes better at this task.

Hugging Face's Trainer handles batching, the forward pass, loss computation, backpropagation and parameter updates for you.
https://huggingface.co/docs/transformers/en/trainer

In [25]:
#Evaluate Fine tuned model
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy
0.294287,0.411372,2,0.820000


{'eval_loss': 0.41137173771858215, 'eval_accuracy': 0.82}


In [26]:
#Savimng  the Fine-Tuned Model
trainer.save_model("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./sentiment_model/tokenizer_config.json', './sentiment_model/tokenizer.json')

Now we have:

sentiment_model/
       ↓
Fine-tuned DistilBERT

This is different from the original:

distilbert-base-uncased

We have adapted it to our specific task.

In [27]:
#Use the Fine-Tuned Model
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="./sentiment_model",
    tokenizer="./sentiment_model"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [28]:
#Testing
sentiment_model("This movie was absolutely fantastic!")

[{'label': 'LABEL_1', 'score': 0.9004030823707581}]

In [29]:
sentiment_model("I hated this movie. It was very boring.")

[{'label': 'LABEL_0', 'score': 0.9057369828224182}]

In [30]:
sentiment_model("The acting was excellent and the story was amazing.")

[{'label': 'LABEL_1', 'score': 0.9184066653251648}]

Label 0 = negative
Label 1 - positive

In [31]:
#Extra info
while True:

    review = input("Enter a movie review (or type 'quit'): ")

    if review.lower() == "quit":
        break

    result = sentiment_model(review)

    print(result)

Enter a movie review (or type 'quit'): the movie was not so good but leo was good
[{'label': 'LABEL_1', 'score': 0.7756021618843079}]
Enter a movie review (or type 'quit'): not a good moive
[{'label': 'LABEL_0', 'score': 0.6615628004074097}]
Enter a movie review (or type 'quit'): a good movie was seen
[{'label': 'LABEL_1', 'score': 0.7800807952880859}]
Enter a movie review (or type 'quit'): quit


As we have fine tuned the distalled model the student can now use this model for sentiment analysis .